# Deteccion Sitios Web Fraudulentos

In [2]:
import requests
import tldextract
import whois
import random
import re
from fake_useragent import UserAgent
from faker import Faker
from datetime import datetime
from tqdm.notebook import tqdm
import pandas as pd
from io import StringIO

# Instancias globales
fake = Faker()
ua = UserAgent()

### Ingesta de Datos Inicial

Inicialmente, se plantea crear 3 datasets distintos:
1. Datasets con sitios web reales: sitios web confiables (no phishing) que se pueden emplear como base comparativa.
2. Dataset con sitios web de la Argentina: sitios de interes a analizar potencial phishing.
3. Dataset con datos sinteticos: sitios falsos para enriquecimiento del dataset de entrenamiento y balancear caracteristicas de sitios web reales y phishing

Se define una funcion auxiliar para generacion del dataset que define las caracteristicas de URLs/dominios de interes a analizar. 

Caracteristicas:

1. domain (str):
2. tld:
3. is_https:
4. url_length:
5. num_hyphens:
6. num_digits
7. num_special_chars
8. num_path_segments
9. has_ssl
10. whois_registrar
11. whois_country
12. domain_age_days

Target:

1. is_phishing (bool)



In [ ]:
def obtener_datos_sitio(url):
    """
    Obtiene información sobre un sitio web real:
    Extrae información básica del dominio (TLD, longitud, caracteres especiales).
    Verifica si el sitio usa HTTPS y tiene SSL.
    Obtiene datos de WHOIS (registrador, país, antigüedad del dominio).
    Asigna la etiqueta is_phishing = 0 porque asumimos que los sitios extraídos son legítimos.
    """
    datos = {"domain": url}

    # Extraer información del dominio
    ext = tldextract.extract(url)
    datos["tld"] = f"{ext.suffix}"  # Extrae el TLD (ej: .com.ar, .gob.ar)

    # Verificar si usa HTTPS
    if url.startswith("https"):
        datos["is_https"] = 1
    else:
        datos["is_https"] = 0

    # Longitud de la URL
    datos["url_length"] = len(url)

    # Contar caracteres especiales
    datos["num_hyphens"] = url.count("-")
    datos["num_digits"] = sum(c.isdigit() for c in url)
    datos["num_special_chars"] = len(re.findall(r"[@#?$%^&*]", url))
    datos["num_path_segments"] = url.count("/")

    # Intentar obtener el certificado SSL
    try:
        response = requests.get(f"https://{url}", headers={"User-Agent": ua.random}, timeout=5)
        if response.status_code == 200:
            datos["has_ssl"] = 1
        else:
            datos["has_ssl"] = 0
    except requests.exceptions.RequestException:
        datos["has_ssl"] = 0

    # Obtener información WHOIS
    try:
        whois_info = whois.whois(url)
        datos["whois_registrar"] = whois_info.registrar if whois_info.registrar else "Desconocido"
        datos["whois_country"] = whois_info.country if whois_info.country else "Desconocido"
        if whois_info.creation_date:
            if isinstance(whois_info.creation_date, list):
                creation_date = whois_info.creation_date[0]
            else:
                creation_date = whois_info.creation_date
            datos["domain_age_days"] = (datetime.now() - creation_date).days
        else:
            datos["domain_age_days"] = -1
    except:
        datos["whois_registrar"] = "Error"
        datos["whois_country"] = "Error"
        datos["domain_age_days"] = -1

    # Asumimos que todos los sitios reales son legítimos
    datos["is_phishing"] = 0

    return datos


#### Lista sitios web reales de *Tranco*

Se opto por emplear el sitio web [*Tranco*](https://tranco-list.eu/) que rankea los sitios web a nivel mundial con mayor trafico y que se actualiza diaramente

In [4]:
# URL of the CSV file to download
csv_url = "https://tranco-list.eu/download/X45KN/1000000"
lista_sitios_web_file = "lista_sitios_web.csv"
dataset_sitios_reales_file = "dataset_sitios_reales.csv"

In [ ]:
# Descargar el archivo CSV
response = requests.get(csv_url)
response.raise_for_status()

# Parsear el contenido CSV
csv_content = StringIO(response.text)

# Crear dataframe y añadir encabezados al CSV
headers = ["rank_position", "url"]
df = pd.read_csv(csv_content, header=None, names=headers)

# Quitar la primer columna que es un índice
df = df.iloc[:, 1:]

print(df.head())

# Guardar como archivo CSV
df.to_csv(lista_sitios_web_file, index=False)

print(f"Arhivo descargado y guardado como: '{lista_sitios_web_file}'")



             url
0     google.com
1        mail.ru
2  microsoft.com
3   facebook.com
4      apple.com
Arhivo descargado y guardado como: 'dataset_sitios_reales.csv'


Crear dataset a partir del la listia de sitios obtenida

In [6]:
NOMBRE_COLUMNA_URLS = headers[1]
LIMIT_ROWS = 100

df_input = pd.read_csv(dataset_sitios_reales_file)

# Convertir la columna de dominios en una lista
sitios_a_procesar = df_input[NOMBRE_COLUMNA_URLS].dropna().tolist()

# Limitar la cantidad de sitios a procesar
sitios_a_procesar = sitios_a_procesar[:LIMIT_ROWS]

datos_reales = []

# Usamos tqdm para una barra de progreso, ideal para listas largas.
# El proceso puede tardar varios minutos (o más) para 3200 registros.
for sitio in tqdm(sitios_a_procesar, desc="Extrayendo datos de sitios web"):
    try:
        datos_sitio = obtener_datos_sitio(sitio)
        datos_reales.append(datos_sitio)
    except Exception as e:
        print(f"❌ Error irrecuperable con {sitio}: {e}. Saltando al siguiente.")
        # Opcional: registrar los errores en la lista también
        # datos_reales.append({'domain': sitio, 'error': str(e)})

# Convertir la lista de resultados en un DataFrame
df_reales = pd.DataFrame(datos_reales)

# Mostrar las primeras filas y la forma del DataFrame resultante
print("\n✅ Proceso completado.")
print(f"Se procesaron {len(df_reales)} registros.")
print("\nPrimeras 5 filas del DataFrame resultante:")
print(df_reales.head())

df_reales.to_csv(dataset_sitios_reales_file, index=False)
print(f"\nResultados guardados exitosamente en: '{dataset_sitios_reales_file}'")


Extrayendo datos de sitios web:   0%|          | 0/100 [00:00<?, ?it/s]

2025-06-29 10:23:16,741 - whois.whois - ERROR - Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
2025-06-29 10:25:07,564 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out



✅ Proceso completado.
Se procesaron 100 registros.

Primeras 5 filas del DataFrame resultante:
          domain  tld  is_https  url_length  num_hyphens  num_digits  \
0     google.com  com         0          10            0           0   
1        mail.ru   ru         0           7            0           0   
2  microsoft.com  com         0          13            0           0   
3   facebook.com  com         0          12            0           0   
4      apple.com  com         0           9            0           0   

   num_special_chars  num_path_segments  has_ssl           whois_registrar  \
0                  0                  0        1         MarkMonitor, Inc.   
1                  0                  0        1              RU-CENTER-RU   
2                  0                  0        1         MarkMonitor, Inc.   
3                  0                  0        0        RegistrarSafe, LLC   
4                  0                  0        1  NOM-IQ Ltd dba Com Laude   

  

#### Web Scraping de *Argendir*

Para la obtencion la creacion del dataset de sitios webs argentinos se empleara el sitio web Argendir, realizando webscrapping para obtener las paginas disponibles sobre un subconjunto de categorias en las que se divide el directorio

In [9]:
#Este código está diseñado para recibir una **lista de categorías** y procesarlas todas en una sola ejecución, guardando todo en un único archivo CSV. Además, incluye mejoras importantes como el manejo de errores y la conversión de URLs relativas a absolutas.

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin # Librería para construir URLs completas

# Lista de categorías que quieres scrapear. ¡Solo tienes que modificar esta lista!
categorias_a_scrapear = ["Educacion", "Gobierno", "Salud", "Noticias-y-medios"]

# URL base del sitio web
url_base = "https://www.argendir.com/"

# Cabecera para simular un navegador y evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Lista para almacenar TODOS los datos de TODAS las categorías
datos_totales = []

print("🚀 Iniciando scraping...")

# Bucle para recorrer cada categoría de la lista
for categoria in categorias_a_scrapear:
    # Construimos la URL completa para la categoría actual
    url_categoria = urljoin(url_base, f"{categoria}/")
    print(f"Procesando categoría: '{categoria}' desde {url_categoria}")

    try:
        # Hacemos la solicitud a la página
        response = requests.get(url_categoria, headers=headers, timeout=10)
        response.raise_for_status()  # Esto generará un error si la página no carga (ej. error 404)

        soup = BeautifulSoup(response.text, "html.parser")

        # Extraer los sitios web de la categoría actual
        sitios_encontrados = soup.select("a.link")
        if not sitios_encontrados:
            print(f"  -> No se encontraron sitios en la categoría '{categoria}'.")
            continue # Pasa a la siguiente categoría

        for sitio in sitios_encontrados:
            nombre = sitio.text.strip()

            # Las URLs en la página son relativas (ej. /sitio/anses.html).
            # Las convertimos en absolutas (ej. https://www.argendir.com/sitio/anses.html)
            url_relativa = sitio["href"]
            url_absoluta = urljoin(url_base, url_relativa)

            descripcion_tag = sitio.find_next("p")
            descripcion = descripcion_tag.text.strip() if descripcion_tag else "Sin descripción"

            # Añadimos los datos usando la variable 'categoria' del bucle
            datos_totales.append([nombre, url_absoluta, categoria, descripcion])

    except requests.exceptions.RequestException as e:
        print(f"  ❌ Error al procesar la categoría '{categoria}': {e}")


# GUARDAR RESULTADOS ---
if datos_totales:
    # Guardar todos los datos en un único CSV
    df_argendir = pd.DataFrame(datos_totales, columns=["Nombre", "URL", "Categoría", "Descripción"])
    df_argendir.to_csv("sitios_argentinos_multiples_categorias.csv", index=False, encoding="utf-8")

    print(f"\n✅ ¡Scraping completado! Se guardaron {len(datos_totales)} sitios en total.")
    print("Archivo generado: sitios_argentinos_multiples_categorias.csv")
else:
    print("\n⚠️ No se pudo extraer ningún dato.")

🚀 Iniciando scraping...
Procesando categoría: 'Educacion' desde https://www.argendir.com/Educacion/
Procesando categoría: 'Gobierno' desde https://www.argendir.com/Gobierno/
Procesando categoría: 'Salud' desde https://www.argendir.com/Salud/
Procesando categoría: 'Noticias-y-medios' desde https://www.argendir.com/Noticias-y-medios/

✅ ¡Scraping completado! Se guardaron 37 sitios en total.
Archivo generado: sitios_argentinos_multiples_categorias.csv


#### Generacion de Datos Sinteticos

Teniendo en cuenta que los datasets previos pueden estar desbalanceados (sesgo de datos) al poseer caracteristicas de sitios web inseguros reducidas, es recomendable enriquecer el dataset mismo empleando datos sinteticos

In [20]:
NUMERO_REGISTROS_SINTETICOS = 5000
PATH_DATASET_SINTETICO = "dataset_sintetico.csv"

In [ ]:
# Lista de TLDs personalizados
TLDS_CHOICES = ['.com', '.org', '.net', '.ar', '.edu', '.gov', '.info', '.io', '.dev', '.biz']

def generate_custom_url():
    protocol = random.choice(["http", "https"])
    domain_name = fake.domain_word()
    tld = random.choice(TLDS_CHOICES)
    return f"{protocol}://{domain_name}{tld}"

# Función para generar un sitio web sintético
def generar_datos_sinteticos(n=100):
    datos_sinteticos = []
    for _ in range(n):
        url = generate_custom_url()  # Genera un dominio aleatorio
        datos_sinteticos.append({
            "domain": url,
            "tld": tldextract.extract(url).suffix,
            "is_https": 1 if url.startswith("https") else 0,
            "url_length": len(url),
            "num_hyphens": url.count("-"),
            "num_digits": sum(c.isdigit() for c in url),
            "num_special_chars" : len(re.findall(r"[@#?$%^&*]", url)),
            "num_path_segments" :url.count("/"),
            "has_ssl" : random.choice([0, 1]),
            "whois_registrar": fake.company(),
            "whois_country": fake.country(),
            "domain_age_days": (datetime.now().date() - fake.date_between(start_date='-20y', end_date='-1y')).days,
            "is_phishing": random.choice([True, False])  # Etiqueta para el modelo
        })

    return pd.DataFrame(datos_sinteticos)

# Generar 
df_sintetico = generar_datos_sinteticos(n=NUMERO_REGISTROS_SINTETICOS)

df_sintetico.head()

,domain,tld,is_https,url_length,num_hyphens,num_digits,num_special_chars,num_path_segments,has_ssl,whois_registrar,whois_country,domain_age_days,is_phishing
0,http//ali-spencer.io,,0,20,1,0,0,2,0,Johnson and Sons,Saint Vincent and the Grenadines,1732,False
1,https//russell.edu,,1,18,0,0,0,2,1,Castillo Group,Afghanistan,1079,True
2,https//jones-wilson.info,,1,24,1,0,0,2,1,Reyes and Sons,Jersey,6281,True
3,https//duran.biz,,1,16,0,0,0,2,0,White-Warner,Malta,1384,True
4,http//stevenson.ar,,0,18,0,0,0,2,1,"Cobb, Evans and Carter",British Indian Ocean Territory (Chagos Archipe...,1327,True


In [22]:
df_sintetico.to_csv(PATH_DATASET_SINTETICO, index=False, encoding="utf-8")

### Preprocesamiento Estructural

### Analisis Exploratorio de Datos (EDA)

### Scraping Complementario Inicial

### Visualizacion